In [ ]:
"""
Week 3 - Day 1
LightGBM + SMOTE Training
==========================
5-fold CV with SMOTE inside folds only.
Target: Macro F1 >= 0.85

Infotact DS/ML Internship — Project 1
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

import sys
import os

sys.path.append("../src")

from external_data.data_fusion import (
    create_fused_dataset,
    get_fused_arrays
)

from lgbm_smote_pipeline import run_lgbm_smote_cv

plt.style.use("seaborn-v0_8")

print("✅ Week 3 modules loaded!")
print("Target Macro F1 : 0.85")

In [ ]:
print("=" * 60)
print("Loading fused dataset...")
print("=" * 60)

fused_df = create_fused_dataset("../data/ai4i2020.csv")

X, y, feature_names = get_fused_arrays(fused_df)

print("\nData ready for modeling!")

print("Shape :", X.shape)
print("Features :", len(feature_names))
print("Failure Rate :", round(y.mean()*100,2), "%")

In [ ]:
print("Starting LightGBM + SMOTE training...")
print("SMOTE applied INSIDE folds only\n")

results = run_lgbm_smote_cv(
    X,
    y,
    feature_names,
    verbose=True
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
folds = range(1, 6)

# F1 per fold
axes[0].bar(
    folds,
    results['fold_f1'],
    color='steelblue',
    edgecolor='black'
)
axes[0].axhline(
    y=results['mean_f1'],
    color='red',
    linestyle='--',
    label=f"Mean={results['mean_f1']:.4f}"
)
axes[0].axhline(
    y=0.85,
    color='green',
    linestyle=':',
    label='Target=0.85'
)
axes[0].set_title('Macro F1 per Fold',
                  fontweight='bold')
axes[0].set_xlabel('Fold')
axes[0].set_ylabel('Macro F1')
axes[0].set_ylim(0, 1)
axes[0].legend()
for i, v in enumerate(results['fold_f1']):
    axes[0].text(
        i+1, v+0.01,
        f'{v:.3f}',
        ha='center',
        fontsize=9,
        fontweight='bold'
    )

# AUC per fold
axes[1].bar(
    folds,
    results['fold_auc'],
    color='coral',
    edgecolor='black'
)
axes[1].axhline(
    y=results['mean_auc'],
    color='red',
    linestyle='--',
    label=f"Mean={results['mean_auc']:.4f}"
)
axes[1].set_title('ROC AUC per Fold',
                  fontweight='bold')
axes[1].set_xlabel('Fold')
axes[1].set_ylabel('ROC AUC')
axes[1].set_ylim(0, 1)
axes[1].legend()
for i, v in enumerate(results['fold_auc']):
    axes[1].text(
        i+1, v+0.005,
        f'{v:.3f}',
        ha='center',
        fontsize=9,
        fontweight='bold'
    )

plt.suptitle(
    'LightGBM + SMOTE — 5-Fold CV Results',
    fontsize=14,
    fontweight='bold'
)
plt.tight_layout()
plt.savefig('../src/week3_cv_results.png')
plt.show()
print("✅ CV results plot saved!")

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns

cm = confusion_matrix(
    results['all_y_true'],
    results['all_y_pred']
)

plt.figure(figsize=(7, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=['No Failure', 'Failure'],
    yticklabels=['No Failure', 'Failure'],
    linewidths=0.5
)
plt.title(
    'Confusion Matrix — All Folds Combined',
    fontweight='bold'
)
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.savefig('../src/week3_confusion_matrix.png')
plt.show()

# Extract values
tn = cm[0][0]
fp = cm[0][1]
fn = cm[1][0]
tp = cm[1][1]

print(f"\n  True Negatives  (Correct No Failure): {tn}")
print(f"  False Positives (False Alarm)        : {fp}")
print(f"  False Negatives (Missed Failure)     : {fn}")
print(f"  True Positives  (Caught Failure)     : {tp}")
print(f"\n  ⚠️  Missed Failures (fn): {fn} — "
      f"minimize this in Week 4!")

In [ ]:
achieved = results['mean_f1'] >= 0.85

print("╔══════════════════════════════════════════╗")
print("║    WEEK 3 DAY 1 — TRAINING SUMMARY       ║")
print("╠══════════════════════════════════════════╣")
print(f"║  Macro F1 Mean  : "
      f"{results['mean_f1']:.4f}"
      f"{'':<19} ║")
print(f"║  Macro F1 Std   : "
      f"{results['std_f1']:.4f}"
      f"{'':<19} ║")
print(f"║  ROC AUC Mean   : "
      f"{results['mean_auc']:.4f}"
      f"{'':<19} ║")
print(f"║  Precision      : "
      f"{results['mean_precision']:.4f}"
      f"{'':<19} ║")
print(f"║  Recall         : "
      f"{results['mean_recall']:.4f}"
      f"{'':<19} ║")
print("╠══════════════════════════════════════════╣")
if achieved:
    print("║  ✅ TARGET F1 >= 0.85 ACHIEVED!          ║")
else:
    print("║  ⚠️  Below target — tune tomorrow!        ║")
print("╠══════════════════════════════════════════╣")
print("║  Tomorrow → Hyperparameter Tuning 🔧     ║")
print("╚══════════════════════════════════════════╝")